## Imports og tilkobling

In [1]:
import sys
import os
import pandas as pd
import io
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient

sys.path.append(os.path.abspath("../../../"))
from src.feature_engineering.forbruksdata import create_consumption_features

load_dotenv()

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")

blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_name = "rawdata"

## Hente CSV fra blob

In [2]:
blob_name = "HARTEVATN.csv"

blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob=blob_name
)

data = blob_client.download_blob().readall()

df = pd.read_csv(io.BytesIO(data))

## Feature engineering

In [3]:
# feature engineering
df = create_consumption_features(df)

# se resultat
df.head()

,metering_point_anonymous,timestamp,value_kwh,transformer_station,consumption_code,hour,weekday,month,is_weekend,is_holiday,day_night
0,4c22dc3e-ce91-4275-9dfd-22ce2f386072,2025-12-25 23:00:00,1.34,HARTEVATN,36,23,3,12,False,False,night
1,4c22dc3e-ce91-4275-9dfd-22ce2f386072,2025-12-26 00:00:00,1.57,HARTEVATN,36,0,4,12,False,False,night
2,4c22dc3e-ce91-4275-9dfd-22ce2f386072,2025-12-26 01:00:00,1.22,HARTEVATN,36,1,4,12,False,False,night
3,4c22dc3e-ce91-4275-9dfd-22ce2f386072,2025-12-26 02:00:00,1.45,HARTEVATN,36,2,4,12,False,False,night
4,4c22dc3e-ce91-4275-9dfd-22ce2f386072,2025-12-26 03:00:00,1.56,HARTEVATN,36,3,4,12,False,False,night


## Legge til blob

In [4]:
import io

# --- Lag CSV i minnet ---
csv_buffer = io.BytesIO()
df.to_csv(csv_buffer, index=False)
csv_buffer.seek(0)  # gå til starten av bufferet

# --- Last opp til blob ---
container_name = "processeddata"
output_blob_name = "HARTEVATN_processed.csv" 
blob_client = blob_service_client.get_blob_client(container=container_name, blob=output_blob_name)
blob_client.upload_blob(csv_buffer, overwrite=True)  # overwrite=True overskriver eksisterende fil

print(f"{output_blob_name} lastet opp til blob!")

HARTEVATN_processed.csv lastet opp til blob!
